In [2]:

# Practical 6: Reinforcement Learning

# 1. Setting up the Maze Environment

import numpy as np
import random

class Maze:
    def __init__(self, size=4):
        self.size = size
        self.reset()

    def reset(self):
        self.agent_pos = [0, 0]          # Start at top-left
        self.goal = [self.size-1, self.size-1]  # Goal at bottom-right
        return tuple(self.agent_pos)

    def step(self, action):
        if action == 0 and self.agent_pos[0] > 0:  # up
            self.agent_pos[0] -= 1

        elif action == 1 and self.agent_pos[0] < self.size-1:  # down
            self.agent_pos[0] += 1

        elif action == 2 and self.agent_pos[1] > 0:  # left
            self.agent_pos[1] -= 1

        elif action == 3 and self.agent_pos[1] < self.size-1:  # right
            self.agent_pos[1] += 1

        if self.agent_pos == self.goal:
            return tuple(self.agent_pos), 1, True

        return tuple(self.agent_pos), -0.01, False


# 2. Defining the Q-Learning Agent

class QLearningAgent:
    def __init__(self, actions, lr=0.1, gamma=0.9, eps=1.0,
                 decay=0.995):
        self.Q = {}
        self.actions = actions
        self.lr = lr
        self.gamma = gamma
        self.eps = eps
        self.decay = decay

    def get_state(self, pos):
        return tuple(pos)

    def choose_action(self, state):
        if random.random() < self.eps:  # exploration
            return random.choice(self.actions)

        self.Q.setdefault(state, np.zeros(len(self.actions)))
        return np.argmax(self.Q[state])

    def learn(self, s, a, r, s2):
        self.Q.setdefault(s, np.zeros(len(self.actions)))
        self.Q.setdefault(s2, np.zeros(len(self.actions)))

        self.Q[s][a] += self.lr * (
            r + self.gamma * np.max(self.Q[s2])
            - self.Q[s][a]
        )

        self.eps *= self.decay


# 3. Training the Agent

env = Maze(size=4)
agent = QLearningAgent(actions=[0,1,2,3])  # up, down, left, right

episodes = 500

for _ in range(episodes):
    state = env.reset()
    done = False

    while not done:
        action = agent.choose_action(state)

        next_state, reward, done = env.step(action)

        agent.learn(state, action, reward, next_state)

        state = next_state


# 4. Testing the Agent

print("\nTesting the trained agent...")

state = env.reset()
done = False
path = [state]

while not done:
    action = agent.choose_action(state)

    next_state, reward, done = env.step(action)

    path.append(next_state)
    state = next_state

print("Path taken by agent:", path)




Testing the trained agent...
Path taken by agent: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (3, 3)]
